# Backtesting (WORKFLOW Stage 3)

Interactive/visual exploration of a backtest — equity curve, cost sensitivity, parameter sweeps. The **reproducible gate** is `uv run scripts/strategy.py validate <name>`; this notebook visualises what that verdict summarises.

> **Parity rule:** this notebook is an *exploration front-end* — it only imports from `mt5_trader.*` and never reimplements signal/engine logic. The same `signal()` runs here, in the backtester, and live. Keep strategy logic causal (no `.shift(-1)`); `forward_return` below is measurement-only.

Run **Kernel → Restart & Run All** before trusting a result. Runs on real ticks (`data/processed`) if present, else deterministic synthetic bars.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from mt5_trader.backtest.engine import BTConfig, run
from mt5_trader.pipeline.data import load_or_synthetic, bars_per_year
from mt5_trader.pipeline import validate
from mt5_trader.strategies import REGISTRY

SYMBOL, EVERY = "XAUUSD", "5m"
BYR = bars_per_year(EVERY)

## 1. Load bars

In [ ]:
bars, is_real = load_or_synthetic(SYMBOL, EVERY)
print(("REAL" if is_real else "SYNTHETIC"), "bars:", bars.shape)

## 2. Run a backtest

In [ ]:
name = "vwap_trend"
res = run(bars, REGISTRY[name]().signal(bars), BTConfig(bars_per_year=BYR))
res.metrics

## 3. Equity curve
Eyeball it — one lucky day is not an edge.

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(res.bars['ts'], res.bars['equity'])
plt.title(f'{name} equity'); plt.show()

## 4. Cost stress
Still positive at 2–4x slippage? Comp fills are unknown until paper.

In [ ]:
slips = [0.5, 1.0, 2.0, 4.0]
rets = [run(bars, REGISTRY[name]().signal(bars),
            BTConfig(slippage_bps=s, bars_per_year=BYR)).metrics['total_return']
        for s in slips]
plt.figure(figsize=(6, 3)); plt.bar([str(s) for s in slips], rets)
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('slippage x'); plt.ylabel('total return')
plt.title(f'{name} cost stress'); plt.show()

## 5. Parameter sweep
Plateau = robust, cliff = overfit. (each strategy has its own primary param to sweep.)

In [ ]:
param, vals = 'entry_n', [120, 180, 240, 300, 360]
sweep = {v: run(bars, REGISTRY[name](**{param: v}).signal(bars),
                BTConfig(bars_per_year=BYR)).metrics['total_return'] for v in vals}
plt.figure(figsize=(6, 3)); plt.plot(list(sweep), list(sweep.values()), marker='o')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel(param); plt.ylabel('total return')
plt.title(f'{name} param sweep'); plt.show()

## 6. Structured verdict
The same gates the CLI runs.

In [ ]:
v = validate.validate_candidate(name, bars, EVERY,
                                wiggle=(param, [180, 240, 300]), n_trials=1)
print('overall:', 'PASS' if v['overall'] else 'FAIL')
for g, d in v['gates'].items():
    flags = ' '.join(f'{k}={val}' for k, val in d.items() if k != 'pass')
    print(f"  [{'ok' if d['pass'] else 'XX'}] {g}: {flags}")

## Graduate
On a clean PASS over **real** data, deploy with `uv run scripts/strategy.py promote <name> --weight <w>`. On a FAIL, `stash add <name> --improve "..."` and iterate.